# [9.4] White-box Evals and Monitors - Solutions

This notebook runs the reference implementation, the visible tests, and direct checks on the committed CUDA verification report.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter9_alignment_interpretability"
section = "part4_white_box_evals_monitors"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_white_box_evals_monitors.solutions as solutions
import part4_white_box_evals_monitors.tests as tests

In [ ]:
tests.test_monitor_dashboard_row_preserves_review_fields(solutions.monitor_dashboard_row)
tests.test_binary_auroc_counts_ties_and_validates_inputs(solutions.binary_auroc)
tests.test_monitor_calibration_report_matches_reference(solutions.monitor_calibration_report)
tests.test_missed_failure_report_identifies_white_box_only_catches(solutions.missed_failure_report)
tests.test_false_positive_documentation_requires_notes(solutions.false_positive_documentation_report)
tests.test_feature_explanation_validation_uses_heldout_accuracy(
    solutions.feature_explanation_validation_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
smoke = solutions.run_smoke_test(cpu=True)
assert smoke["dashboard"]["active_features"] == ("summary", "benign")
assert smoke["calibration"]["auroc"] == 1.0
assert smoke["missed_failure"]["caught_failure_indices"] == (0,)
assert smoke["false_positive"]["documented"]
assert smoke["explanation_validation"]["heldout_accuracy"] == 1.0
smoke

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

assert report["accepted"] and report["tests_passed"]
assert report["gt_tier"] == "GT-3"
assert report["notebook_id"] == "9_4_white_box_evals_and_monitors"
assert gpu["cuda_available"] and gpu["preflight_passed"]
assert gpu["model_name"] == "EleutherAI/pythia-70m-deduped"
assert gpu["hf_revision"] == "e93a9faa9c77e5d09219f6c868bfc7a1bd65593c"
assert gpu["train_prompt_count"] == 36
assert gpu["heldout_prompt_count"] == 24
assert gpu["failure_kind_count"] == 5
assert gpu["monitor_auroc"] == 1.0
assert gpu["white_box_accuracy"] == 1.0
assert gpu["black_box_proxy_accuracy"] == 0.875
assert gpu["black_box_missed_failure_count"] >= 1
assert gpu["catches_black_box_miss"]
assert gpu["label_shuffled_monitor_auroc"] <= 0.85
assert gpu["random_direction_monitor_auroc"] <= 0.85
assert gpu["false_positives_documented"]
assert gpu["explanations_validated"]
assert not gpu["generation_used"]
assert gpu["peak_vram_gb"] < 1.0
assert gpu["within_vram_budget"]

tests.test_committed_gpu_report_matches_white_box_monitor_contract(gpu)
{
    "device": gpu["device"],
    "monitor_auroc": gpu["monitor_auroc"],
    "black_box_proxy_accuracy": gpu["black_box_proxy_accuracy"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}